# Первый этап задания RAG Playground

Ноутбук показывает весь поток первого этапа: чтение настроек, подготовку SciFact, проверку целостности, health checks и запуск автотестов.

Ноутбук не печатает credentials GigaChat. Он также не скачивает датасет повторно, если локальные файлы уже существуют.

## 1. Поиск корня проекта и подготовка импортов

Ячейка находит `pyproject.toml`, поэтому ноутбук можно запускать из корня или из папки `notebooks`.

In [1]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

# Ищем ближайший родительский каталог с pyproject.toml.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден корень проекта с pyproject.toml')

# src добавляется в путь импорта для работы даже без editable-установки.
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Корень проекта: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')
print(f'Исполняемый Python: {sys.executable}')

Корень проекта: /Users/e.ilyina/Desktop/Портфолио/RAG
Python: 3.13.0
Исполняемый Python: /Users/e.ilyina/Desktop/Портфолио/RAG/.venv/bin/python


## 2. Чтение конфигурации

Здесь проверяем, какие URL, seed и embedding-модель будут использованы. Вместо самого ключа GigaChat выводится только факт его наличия.

In [2]:
from rag_playground.config import Settings

# Читаем .env именно из корня, независимо от cwd ядра.
settings = Settings.from_env(PROJECT_ROOT / '.env')
safe_settings = {
    'data_dir': str(PROJECT_ROOT / settings.data_dir),
    'random_seed': settings.random_seed,
    'qdrant_url': settings.qdrant_url,
    'elasticsearch_url': settings.elasticsearch_url,
    'gigachat_credentials_configured': bool(settings.gigachat_credentials),
    'gigachat_scope': settings.gigachat_scope,
    'gigachat_base_url': settings.gigachat_base_url,
    'gigachat_verify_ssl_certs': settings.gigachat_verify_ssl_certs,
    'gigachat_embedding_model': settings.gigachat_embedding_model,
}
print(json.dumps(safe_settings, ensure_ascii=False, indent=2))

{
  "data_dir": "/Users/e.ilyina/Desktop/Портфолио/RAG/data",
  "random_seed": 42,
  "qdrant_url": "http://localhost:6333",
  "elasticsearch_url": "http://localhost:9200",
  "gigachat_credentials_configured": false,
  "gigachat_scope": "GIGACHAT_API_PERS",
  "gigachat_base_url": "https://api.giga.chat/v1",
  "gigachat_verify_ssl_certs": true,
  "gigachat_embedding_model": "EmbeddingsGigaR"
}


## 3. Загрузка и подготовка SciFact

По умолчанию датасет загружается, только если локального манифеста ещё нет. Для принудительной повторной загрузки измените `FORCE_DOWNLOAD` на `True`.

In [3]:
from rag_playground.data import prepare_scifact

FORCE_DOWNLOAD = False  # True принудительно перезагрузит данные с Hugging Face.
data_root = PROJECT_ROOT / settings.data_dir
manifest_path = data_root / 'raw' / 'scifact' / 'manifest.json'

if FORCE_DOWNLOAD or not manifest_path.exists():
    print('Загружаем и нормализуем SciFact...')
    manifest = prepare_scifact(data_root, split='test')
else:
    print('Локальные данные уже подготовлены; читаем существующий манифест.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print(json.dumps(manifest, ensure_ascii=False, indent=2))

Локальные данные уже подготовлены; читаем существующий манифест.
{
  "corpus_documents": 5183,
  "dataset": "BeIR/scifact",
  "qrels": 339,
  "qrels_dataset": "BeIR/scifact-qrels",
  "queries": 1109,
  "split": "test"
}


## 4. Примеры подготовленных данных

Читаем по одной записи из корпуса, запросов и qrels. Длинный текст документа сокращается только для экрана; в файле он сохранён полностью.

In [4]:
scifact_dir = data_root / 'raw' / 'scifact'

# JSONL читается построчно: не нужно загружать весь корпус для одного примера.
with (scifact_dir / 'corpus.jsonl').open(encoding='utf-8') as file:
    sample_document = json.loads(next(file))
with (scifact_dir / 'queries.jsonl').open(encoding='utf-8') as file:
    sample_query = json.loads(next(file))
with (scifact_dir / 'qrels' / 'test.tsv').open(encoding='utf-8') as file:
    sample_qrel = next(csv.DictReader(file, delimiter='\t'))

document_preview = dict(sample_document)
document_preview['text'] = document_preview['text'][:300] + '...'
print('Документ:', json.dumps(document_preview, ensure_ascii=False, indent=2))
print('\nЗапрос:', json.dumps(sample_query, ensure_ascii=False, indent=2))
print('\nQrel:', json.dumps(sample_qrel, ensure_ascii=False, indent=2))

Документ: {
  "doc_id": "4983",
  "text": "Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent di...",
  "title": "Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging."
}

Запрос: {
  "query_id": "0",
  "text": "0-dimensional biomaterials lack inductive properties."
}

Qrel: {
  "query_id": "1",
  "doc_id": "31715818",
  "score": "1"
}


## 5. Проверка целостности SciFact

Важно не только посчитать строки, но и убедиться, что каждый `query_id` и `doc_id` из qrels ссылается на существующую запись.

In [5]:
# Для быстрой проверки собираем множества ID. Для SciFact они небольшие.
with (scifact_dir / 'corpus.jsonl').open(encoding='utf-8') as file:
    corpus_ids = {json.loads(line)['doc_id'] for line in file}
with (scifact_dir / 'queries.jsonl').open(encoding='utf-8') as file:
    query_ids = {json.loads(line)['query_id'] for line in file}
with (scifact_dir / 'qrels' / 'test.tsv').open(encoding='utf-8') as file:
    qrels = list(csv.DictReader(file, delimiter='\t'))

missing_documents = {row['doc_id'] for row in qrels} - corpus_ids
missing_queries = {row['query_id'] for row in qrels} - query_ids
non_positive_scores = [row for row in qrels if int(row['score']) <= 0]

assert not missing_documents, f'Не найдены doc_id: {missing_documents}'
assert not missing_queries, f'Не найдены query_id: {missing_queries}'
assert not non_positive_scores, 'В test qrels найдены неположительные score'

print(f'Документов: {len(corpus_ids)}')
print(f'Запросов: {len(query_ids)}')
print(f'Test qrels: {len(qrels)}')
print('Целостность: OK — все ссылки qrels валидны.')

Документов: 5183
Запросов: 1109
Test qrels: 339
Целостность: OK — все ссылки qrels валидны.


## 6. Проверка Qdrant и Elasticsearch

Эта ячейка не запускает Docker-контейнеры автоматически. Она только показывает, доступны ли сервисы по URL из конфигурации.

In [6]:
from rag_playground.health import check_local_services

service_results = check_local_services(settings)
for result in service_results:
    marker = 'OK' if result.ok else 'FAIL'
    print(f'[{marker}] {result.name}: {result.detail}')

if not all(result.ok for result in service_results):
    print('\nЕсли сервисы ещё не запущены, поднимите их через Docker Compose и повторите ячейку.')

[FAIL] Qdrant: unavailable: <urlopen error [Errno 61] Connection refused>
[FAIL] Elasticsearch: unavailable: <urlopen error [Errno 61] Connection refused>

Если сервисы ещё не запущены, поднимите их через Docker Compose и повторите ячейку.


## 7. Проверка GigaChat

Проверка запрашивает список моделей, доступных аккаунту. Она не отправляет промпт и не печатает credentials. Без заполненного `.env` ожидаем понятный `FAIL`.

In [7]:
from rag_playground.health import check_gigachat

gigachat_result = check_gigachat(settings)
marker = 'OK' if gigachat_result.ok else 'FAIL'
print(f'[{marker}] {gigachat_result.name}: {gigachat_result.detail}')

[FAIL] GigaChat: GIGACHAT_CREDENTIALS is not configured


## 8. Автотесты, линтер и проверка Docker Compose

Последние ячейки запускают те же проверки, которые используются при разработке. Вывод сохраняется прямо в ноутбуке.

In [8]:
def run_command(title: str, command: list[str]) -> subprocess.CompletedProcess[str]:
    """Запустить команду и напечатать объединённый stdout/stderr."""
    print(f'=== {title} ===')
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(completed.stdout.rstrip())
    print(f'Код завершения: {completed.returncode}\n')
    return completed

# -m pytest гарантирует, что тесты запускаются тем же Python, что и ядро ноутбука.
test_run = run_command('Тесты Pytest', [sys.executable, '-m', 'pytest'])
if test_run.returncode != 0:
    raise RuntimeError('Тесты завершились с ошибкой')

=== Тесты Pytest ===


........                                                                 [100%]
8 passed in 0.04s
Код завершения: 0



In [9]:
# Ruff проверяет ошибки, импорты и типичные опасные конструкции.
lint_run = run_command('Проверка Ruff', [sys.executable, '-m', 'ruff', 'check', 'src', 'tests'])
if lint_run.returncode != 0:
    raise RuntimeError('Ruff нашёл ошибки')

=== Проверка Ruff ===
All checks passed!
Код завершения: 0



In [10]:
# Docker Compose не запускается: config --quiet только валидирует YAML и параметры.
compose_run = run_command(
    'Валидация Docker Compose',
    ['docker', 'compose', '-f', str(PROJECT_ROOT / 'compose.yaml'), 'config', '--quiet'],
)
if compose_run.returncode != 0:
    print('Docker Compose недоступен или compose.yaml не прошёл валидацию.')

=== Валидация Docker Compose ===



Код завершения: 0



## 9. Что доказывает этот ноутбук

После полного запуска видно, что:

- конфигурация читается без раскрытия секретов;
- SciFact приведён к стабильной локальной схеме;
- qrels ссылаются только на существующие документы и запросы;
- состояние Qdrant, Elasticsearch и GigaChat видно отдельно;
- результаты Pytest, Ruff и валидации Compose сохраняются в выводах ячеек.